# Precommit v2.4 — code-only A100 launcher
Select A100. This iLLaDA exploration tests dependency selection, spatial order, B-first persistence, and token space. Seed **123**; no reserved confirmation inference. Set REVISION to an exact commit for a frozen run. Code only; no historical results.


In [ ]:
from pathlib import Path
import subprocess, sys, json
REVISION = "main"  # Replace with an exact commit for a frozen run.
ROOT = Path("/content/DLM_Multitask_Reasoning")
if ROOT.exists():
    raise FileExistsError("Use a fresh runtime or preserve the existing checkout before a new run.")
subprocess.run(["git", "clone", "https://github.com/Twist-Shan/DLM_Multitask_Reasoning.git", str(ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=ROOT, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Checked out", COMMIT)


## Environment and mechanical checks
The isolated environment inherits Colab CUDA PyTorch while pinning the model-interface dependencies. No model weights are loaded by these tests.


In [ ]:
import os
os.environ.update(USE_TF="0", USE_FLAX="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", str(ROOT / ".colab-env")], check=True)
PY = str(ROOT / ".colab-env/bin/python")
subprocess.run([PY, "-m", "pip", "install", "-e", str(ROOT), "-r", str(ROOT / "requirements/precommit.txt")], check=True)
subprocess.run([PY, "-m", "unittest", "discover", "-s", "tests", "-p", "test_precommit*.py", "-v"], cwd=ROOT, check=True)
subprocess.run([PY, "scripts/package_precommit_code.py", "--out", "/content/precommit-v24-code.zip"], cwd=ROOT, check=True)
RUNS = ROOT / "runs"
RUNS.mkdir(exist_ok=False)
(RUNS / "checkout.json").write_text(json.dumps({"revision_requested": REVISION, "commit": COMMIT}, indent=2))
(RUNS / "environment.freeze.txt").write_bytes(subprocess.check_output([PY, "-m", "pip", "freeze"]))


## Run the bounded A100 exploration
This starts paid GPU computation: one iLLaDA run, with a 5400-second budget. Partial records are archived on failure.


In [ ]:
subprocess.run([PY, "scripts/colab_precommit_v24_suite.py"], cwd=ROOT, check=True, timeout=5600)
status = json.loads((RUNS / "suite_status.json").read_text())
print(json.dumps(status, indent=2))
if status["status"] != "complete":
    raise RuntimeError("Suite incomplete. Preserve and download the partial archive; inspect model logs.")


## Audit saved probabilities
Run after a complete suite. All diagnostic outputs are local to this runtime and contain your own results.


In [ ]:
subprocess.run([PY, "scripts/analyze_precommit_v24.py", str(RUNS / "illada-v24"), "--out", str(RUNS / "analysis.json")], cwd=ROOT, check=True)


## Save outputs, then release the GPU
The final ZIP includes code provenance, raw outputs and your audit files. Download and verify it locally before disconnecting and deleting the runtime. No result is pushed to GitHub. This download cell also works after a failed run if the earlier setup finished.


In [ ]:
import hashlib, shutil
from google.colab import files
archive = Path(shutil.make_archive("/content/precommit-v24-with-audits", "zip", ROOT, "runs"))
print("bytes", archive.stat().st_size, "sha256", hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))
